In [1]:
"""
Algorithm Comparison Script — HP Landslide Project
====================================================
Run: python compare_algorithms.py
Output: prints comparison table + saves algorithm_comparison.csv
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.ensemble import (GradientBoostingClassifier,
                               RandomForestClassifier,
                               AdaBoostClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              precision_score, recall_score,
                              f1_score, confusion_matrix,
                              classification_report)

# ═══════════════════════════════════════════════
# 1. LOAD & PREPARE DATASET
# ═══════════════════════════════════════════════
df = pd.read_csv("C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/data/processed/hp_landslide_dataset_v7_final.csv")

df['soil_moisture'] = df['soil_moisture'].fillna(df['soil_moisture'].median())
df['ndvi']          = df['ndvi'].clip(-1, 1)
df['aspect']        = df['aspect'].fillna(180)
df = df.dropna(subset=['rainfall_7day', 'slope', 'elevation', 'label'])
df = df[df['ndvi'] <= 1.0]

# Balance dataset
df_majority           = df[df['label'] == 0]
df_minority           = df[df['label'] == 1]
df_minority_upsampled = resample(df_minority, replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_balanced = pd.concat([df_majority, df_minority_upsampled])

print("=== Dataset after balancing ===")
print(df_balanced['label'].value_counts())
print(f"Total rows: {len(df_balanced)}\n")

FEATURE_COLS = ['rainfall_7day', 'slope', 'elevation',
                'soil_moisture', 'ndvi', 'aspect']
X = df_balanced[FEATURE_COLS]
y = df_balanced['label']

# Train/test split for detailed metrics
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ═══════════════════════════════════════════════
# 2. DEFINE ALGORITHMS
# ═══════════════════════════════════════════════
algorithms = {
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":        DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=200,
                                                        max_depth=3,
                                                        learning_rate=0.05,
                                                        random_state=42),
    "AdaBoost":             AdaBoostClassifier(n_estimators=200, random_state=42),
    "SVM":                  SVC(probability=True, kernel='rbf', random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32),
                                           max_iter=500, random_state=42),
}

# ═══════════════════════════════════════════════
# 3. CROSS-VALIDATION COMPARISON
# ═══════════════════════════════════════════════
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 85)
print(f"{'Algorithm':<25} {'Accuracy':>10} {'ROC-AUC':>10} {'Precision':>10} {'Recall':>10} {'F1':>8}")
print("=" * 85)

results = []

for name, clf in algorithms.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    clf)
    ])

    acc  = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy').mean()
    auc  = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc').mean()
    prec = cross_val_score(pipe, X, y, cv=cv, scoring='precision').mean()
    rec  = cross_val_score(pipe, X, y, cv=cv, scoring='recall').mean()
    f1   = cross_val_score(pipe, X, y, cv=cv, scoring='f1').mean()

    results.append({
        'Algorithm': name,
        'Accuracy':  round(acc,  4),
        'ROC-AUC':   round(auc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1-Score':  round(f1,   4),
    })

    print(f"{name:<25} {acc:>10.4f} {auc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>8.4f}")

print("=" * 85)

# ═══════════════════════════════════════════════
# 4. SORT & SHOW BEST
# ═══════════════════════════════════════════════
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1  # rank starts from 1

print("\n=== RANKED BY ROC-AUC ===")
print(results_df.to_string())

best = results_df.iloc[0]
print(f"\n BEST MODEL : {best['Algorithm']}")
print(f"  Accuracy  : {best['Accuracy']}")
print(f"  ROC-AUC   : {best['ROC-AUC']}")
print(f"  F1-Score  : {best['F1-Score']}")

# ═══════════════════════════════════════════════
# 5. DETAILED REPORT FOR BEST MODEL
# ═══════════════════════════════════════════════
print(f"\n=== DETAILED REPORT — {best['Algorithm']} ===")

best_clf  = algorithms[best['Algorithm']]
best_pipe = Pipeline([('scaler', StandardScaler()), ('clf', best_clf)])
best_pipe.fit(X_train, y_train)

y_pred  = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

print(f"Test Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['No Landslide','Landslide'])}")

# ═══════════════════════════════════════════════
# 6. SAVE TO CSV
# ═══════════════════════════════════════════════
save_path = "C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/algorithm_comparison.csv"
results_df.to_csv(save_path, index=False)
print(f"\nResults saved to: {save_path}")
print("Use this table directly in your research paper as Table 2")

=== Dataset after balancing ===
label
0    475
1    475
Name: count, dtype: int64
Total rows: 950

Algorithm                   Accuracy    ROC-AUC  Precision     Recall       F1
Logistic Regression           0.7316     0.8012     0.7152     0.7705   0.7417
Decision Tree                 0.8232     0.9219     0.8328     0.8358   0.8218
Random Forest                 0.9726     0.9975     0.9578     0.9895   0.9732
Gradient Boosting             0.9558     0.9861     0.9308     0.9853   0.9571
AdaBoost                      0.8505     0.9377     0.8394     0.8695   0.8533
SVM                           0.8432     0.9225     0.8434     0.8421   0.8417
KNN                           0.8379     0.9258     0.7854     0.9305   0.8513
Neural Network (MLP)          0.9505     0.9779     0.9200     0.9895   0.9528

=== RANKED BY ROC-AUC ===
              Algorithm  Accuracy  ROC-AUC  Precision  Recall  F1-Score
1         Random Forest    0.9726   0.9975     0.9578  0.9895    0.9732
2     Gradient Boos